In [124]:
import numpy as np

# а) Гаусс, обратная матрица

In [125]:
def gauss(A, B):
    n = len(B)
    A = A.astype(float)
    B = B.astype(float)

    for k in range(n):
        max_row = k + np.argmax(np.abs(A[k:, k]))
        A[[k, max_row]] = A[[max_row, k]]
        B[[k, max_row]] = B[[max_row, k]]
        
        for i in range(k + 1, n):
            if A[k, k] == 0: continue
            m = A[i, k] / A[k, k]
            A[i, k:] -= m * A[k, k:]
            B[i] -= m * B[k]

    X = np.zeros(n)
    for i in range(n - 1, -1, -1):
        X[i] = (B[i] - np.dot(A[i, i+1:], X[i+1:])) / A[i, i]
    return X

In [126]:
def inv_matrix(A, B):
    A_inv = np.linalg.inv(A)
    return np.dot(A_inv, B)

In [127]:
A_a = np.array([
    [1.84, 0.79, -3.46, 2.73],
    [2.37, -1.62, 0.55, -3.81],
    [3.68, 2.14, -1.36, 0.89],
    [-1.42, 3.51, 2.67, -0.74]
], dtype=float)
B_a = np.array([5.19, -4.72, 4.38, 2.83], dtype=float)

In [128]:
gauss(A_a, B_a)

array([-0.13792276,  1.53838019, -0.93491516,  0.36397475])

In [129]:
inv_matrix(A_a, B_a)

array([-0.13792276,  1.53838019, -0.93491516,  0.36397475])

# б) Якоби, Зейдель (0,01)

In [130]:
def jacobi(A, B, X0, eps=0.01, max_iter=100):
    n = len(B)
    X = X0.copy()
    for i in range(max_iter):
        X_new = np.zeros(n)
        for i in range(n):
            s = sum(A[i, j] * X[j] for j in range(n) if j != i)
            X_new[i] = (B[i] - s) / A[i, i]
        if np.max(np.abs(X_new - X)) < eps:
            print(f'iter: {i}')
            return X_new
        X = X_new
    print('max_iter stop')
    return X

In [131]:
def seidel(A, B, X0, eps=0.01, max_iter=100):
    n = len(B)
    X = X0.copy()
    for i in range(max_iter):
        X_old = X.copy()
        for i in range(n):
            s1 = sum(A[i, j] * X[j] for j in range(i))
            s2 = sum(A[i, j] * X[j] for j in range(i + 1, n))
            X[i] = (B[i] - s1 - s2) / A[i, i]
        if np.max(np.abs(X - X_old)) < eps:
            print(f'iter: {i}')
            return X
    print('max_iter stop')
    return X

In [132]:
A_b = np.array([
    [3, 1, -1, 1],
    [1, -4, 1, -1],
    [-1, 1, 4, 1],
    [1, 2, 1, -5]
], dtype=float)
B_b = np.array([33, 5, 4, 13], dtype=float)
X0_b = np.zeros(4)

In [133]:
jacobi(A_b, B_b, X0_b, eps=0.000000001)

iter: 3


array([11.,  2.,  3.,  1.])

In [134]:
seidel(A_b, B_b, X0_b, eps=0.000000001)

iter: 3


array([11.,  2.,  3.,  1.])

# в) Прогонки

In [135]:
A_c_diag = [10, 2, -7, 5]
A_c_upper = [-4, -0.2, 1, 0]
A_c_lower = [0, 1, 1, -2]
B_c = [8, 5.5, 2, -1]

In [136]:
def progonka(a, b, c, d):
    n = len(d)
    alpha = np.zeros(n)
    beta = np.zeros(n)
    x = np.zeros(n)

    alpha[0] = -c[0] / b[0]
    beta[0] = d[0] / b[0]
    for i in range(1, n):
        denom = a[i] * alpha[i-1] + b[i]
        alpha[i] = -c[i] / denom
        beta[i] = (d[i] - a[i] * beta[i-1]) / denom

    x[n-1] = beta[n-1]
    for i in range(n-2, -1, -1):
        x[i] = alpha[i] * x[i+1] + beta[i]
    return x

In [137]:
progonka(A_c_lower, A_c_diag, A_c_upper, B_c)

array([ 1.58209719,  1.95524297, -0.0370844 , -0.21483376])

# Доп. задание

In [138]:
def qr_decomposition(M):
    n = M.shape[0]
    Q = np.zeros((n, n))
    R = np.zeros((n, n))
    
    for j in range(n):
        v = M[:, j].copy()
        for i in range(j):
            R[i, j] = np.dot(Q[:, i], M[:, j])
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        if R[j, j] > 1e-10:
            Q[:, j] = v / R[j, j]
    return Q, R

In [139]:
def qr_algorithm(M, max_iter=1000, tol=1e-8):
    n = M.shape[0]
    A_cur = M.copy()
    Q_total = np.eye(n)
    
    for _ in range(max_iter):
        Q, R = qr_decomposition(A_cur)
        A_cur = R @ Q
        Q_total = Q_total @ Q

        off_diag = np.sum(np.abs(np.tril(A_cur, -1)))
        if off_diag < tol:
            break
    
    eigenvalues = np.diag(A_cur)

    result_dict = {round(eigenvalues[i], 6): Q_total[:, i] for i in range(n)}
    
    return result_dict

In [140]:
A = np.array([
    [51.15, 2.13, -1.16, 0.15],
    [2.77, 1.15, -0.73, -1.97],
    [3.48, 0.97, 16.21, 9.32],
    [3.76, 0.21, -0.58, 0.76]
], dtype=float)

In [141]:
qr_algorithm(A)

{51.129162: array([0.98891784, 0.05020192, 0.11934107, 0.07265667]),
 16.018373: array([-0.11427094, -0.04652911,  0.9915074 , -0.04111293]),
 0.603803: array([-0.09041418,  0.30954996,  0.04331494,  0.94558337]),
 1.518663: array([-0.02844193,  0.94841638,  0.02818875, -0.31448819])}